In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/belyaevgleb/avito-model-dataset/train_pairs.jsonl
/kaggle/input/datasets/belyaevgleb/avito-model-dataset/val_pairs.jsonl
/kaggle/input/datasets/belyaevgleb/requirements-1/requirements.txt


In [2]:
# # 1. Copy requirements.txt to a writable location
# !cp /kaggle/input/datasets/belyaevgleb/requirements-1/requirements.txt /tmp/requirements.txt

# # 2. Remove the strict faiss-cpu constraint from the temporary file
# !sed -i '/faiss-cpu/d' /tmp/requirements.txt

# # 3. Install all other requirements
# !pip install -r /tmp/requirements.txt

# # 4. Install a compatible version of faiss-cpu separately
# !pip install faiss-cpu

In [3]:
# !pip install transformers accelerate peft datasets sentence-transformers bitsandbytes faiss-cpu
# !pip install -q transformers_stream_generator tiktoken

In [4]:
!pip install --upgrade \
    "transformers>=4.45.0" \
    "tokenizers>=0.20.0" \
    "accelerate>=0.30.0" \
    "peft>=0.12.0" \
    "bitsandbytes>=0.46.1" \
    "datasets>=2.20.0" \
    "sentence-transformers>=2.7.0" \
    "faiss-cpu>=1.9.0" \
    


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 21.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 19.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 87.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.3/608.3 kB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 19.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 133.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 117.3 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Success

In [5]:
!pip install -q -U "pyarrow<24,>=19.0.0" "datasets<4.0.0" "bitsandbytes>=0.46.1" "transformers>=4.45.0" "peft>=0.12.0" "accelerate>=0.30.0" "faiss-cpu>=1.9.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [6]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    total_gb = torch.cuda.mem_get_info()[1] / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Free VRAM: {free_gb:.2f} GB / {total_gb:.2f} GB")

CUDA available: True
GPU: Tesla T4
Free VRAM: 14.46 GB / 14.56 GB


In [7]:
import torch
import gc

# 1. Delete objects/models/tensors taking up space
# (Replace 'model', 'optimizer', etc. with your actual variable names)
try:
    del model
    del optimizer
    del inputs
    del outputs
except NameError:
    pass

# 2. Run Python's garbage collector
gc.collect()

# 3. Clear PyTorch's CUDA cache allocator
torch.cuda.empty_cache()

# 4. Check current VRAM usage
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Cached/Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Allocated: 0.00 GB
Cached/Reserved: 0.00 GB


In [8]:
"""
As i mantioned before i am going to use LoRA for 
fine-tune of Qwen-7B model to use it as agent. In 
that file i will mention configuration and use it \
as studying script on kaggle
"""

import json
from pathlib import Path
from typing import List, Dict, Any
from dataclasses import dataclass


import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,  # type: ignore
    get_peft_model,  # type: ignore
    prepare_model_for_kbit_training,  # type: ignore
)

# configuration for pc

# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# LORA_OUTDIR = "adapters/qwen_lora"
# TRAIN_FILE = "data/processed/train_pairs.jsonl"
# VAL_FILE = "data/processed/val_pairs.jsonl"
# MAX_SEQ_LENGTH = 1024
# BATCH_SIZE = 2
# GRAD_ACC_STEPS = 4
# EPOCHS_OR_STEPS = {"max_steps": 3000}
# LR_LORA = 2e-4
# LR_BASE = 0

# configuration for kaggle

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
LORA_OUTDIR = "/kaggle/working/adapters/qwen_lor"
TRAIN_FILE = "/kaggle/input/datasets/belyaevgleb/avito-model-dataset/train_pairs.jsonl"
VAL_FILE = "/kaggle/input/datasets/belyaevgleb/avito-model-dataset/val_pairs.jsonl"
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 2
GRAD_ACC_STEPS = 4
EPOCHS_OR_STEPS = {"max_steps": 3000}
LR_LORA = 2e-4
LR_BASE = 0


@dataclass
class LoraArguments:
    r: int = 16
    lora_alpha: int = 32
    target_modules = ["q_proj", "v_proj", "gate_proj"]
    lora_dropout = 0.05
    bias = "none"
    task_type = "CASUAL_LM"


def load_jsonl_to_dataset(path: str) -> Dataset:
    """Reads a JSONL file and converts it into a Hugging Face Dataset."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    if not records:
        raise ValueError(f"File {path} is empty or not found.")
    return Dataset.from_dict({k: [rec[k] for rec in records] for k in records[0]})


# ----------------------------------------------------------------------
# 3. ChatML Prompt Construction & Tokenization
# ----------------------------------------------------------------------
def build_chat_prompt(example: Dict[str, Any], tokenizer) -> Dict[str, Any]:
    system_text = ""
    user_text = ""
    assistant_text = ""

    for msg in example["messages"]:
        role = msg.get("role")
        content = msg.get("content", "")
        if role == "system":
            system_text = content
        elif role == "user":
            user_text = content
        elif role == "assistant":
            assistant_text = content

    prompt_str = (
        f"<|im_start|>system\n{system_text}<|im_end|>\n"
        f"<|im_start|>user\n{user_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    response_str = f"{assistant_text}<|im_end|>"

    prompt_ids   = tokenizer.encode(prompt_str,   add_special_tokens=False)
    response_ids = tokenizer.encode(response_str, add_special_tokens=False)

    input_ids = prompt_ids + response_ids
    labels    = [-100] * len(prompt_ids) + response_ids

    # Truncate
    if len(input_ids) > MAX_SEQ_LENGTH:
        input_ids = input_ids[:MAX_SEQ_LENGTH]
        labels    = labels[:MAX_SEQ_LENGTH]

    # ✅ Pad to exactly MAX_SEQ_LENGTH so all sequences have identical length
    pad_id  = tokenizer.pad_token_id
    pad_len = MAX_SEQ_LENGTH - len(input_ids)

    input_ids     = input_ids     + [pad_id] * pad_len
    attention_mask = [1] * (MAX_SEQ_LENGTH - pad_len) + [0] * pad_len
    labels        = labels        + [-100]   * pad_len   # -100 = ignore padding in loss

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         labels,
    }

# ----------------------------------------------------------------------
# 4. Main Training Routine
# ----------------------------------------------------------------------
def main() -> None:
    # 4.1 Tokenizer setup (transformers 4.41.0 compatible)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)
    print(
        f"Tokenizer loaded: pad_token={tokenizer.pad_token} (id={tokenizer.pad_token_id})"
    )
    # 4.2 Dataset loading & preprocessing
    print("Loading datasets...")
    train_ds = load_jsonl_to_dataset(TRAIN_FILE)
    val_ds = load_jsonl_to_dataset(VAL_FILE)
    train_ds = train_ds.map(
        lambda ex: build_chat_prompt(ex, tokenizer),
        remove_columns=train_ds.column_names,
    )
    val_ds = val_ds.map(
        lambda ex: build_chat_prompt(ex, tokenizer),
        remove_columns=val_ds.column_names,
    )
    # 4.3 8-bit Quantization Config (bitsandbytes==0.43.1 standard)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    print("Loading 8-bit base model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
    )
    print("✅ Базовая модель успешно загружена!")
    # Prepare for k-bit training (freezes base model layers, casts layernorms)
    model = prepare_model_for_kbit_training(model)
    # 4.4 LoRA Setup (peft==0.7.1 standard)
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "gate_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    # 4.5 Optimizer Parameter Groups (differentiated learning rate)
    lora_params = [
        p for n, p in model.named_parameters() if "lora" in n and p.requires_grad
    ]
    base_params = [
        p for n, p in model.named_parameters() if "lora" not in n and p.requires_grad
    ]
    optimizer_grouped_parameters = [
        {"params": lora_params, "lr": LR_LORA},
    ]
    if base_params:
        optimizer_grouped_parameters.append({"params": base_params, "lr": LR_BASE})
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, betas=(0.9, 0.999))
    # 4.6 Training Arguments (transformers==4.41.0 uses evaluation_strategy)
    training_args = TrainingArguments(
        output_dir="/kaggle/working/tmp_checkpoints",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC_STEPS,
        max_steps=1200,
        learning_rate=LR_LORA,
        logging_steps=5,  # Log every 5 steps so you see output fast
        save_steps=200,
        save_total_limit=2,
        eval_strategy="no",  # Explicitly disable evaluation
        fp16=True,
        gradient_checkpointing=True,
        report_to="none",
        remove_unused_columns=False,
        dataloader_pin_memory=False,
    )
    # No DataCollatorForSeq2Seq — use the default one that just pads input_ids
    # DataCollatorForSeq2Seq was blocking on eval pre-padding
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        # eval_dataset removed — eval_strategy="no"
        optimizers=(optimizer, None),
    )
    print("Starting training...")
    trainer.train()
    print(f"Saving LoRA adapter to {LORA_OUTDIR}...")
    Path(LORA_OUTDIR).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(LORA_OUTDIR)
    tokenizer.save_pretrained(LORA_OUTDIR)
    print("✅ Training successfully completed.")


if __name__ == "__main__":
    main()


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded: pad_token=<|endoftext|> (id=151643)
Loading datasets...


... (9 outputs left out)

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


trainable params: 15,138,816 || all params: 7,630,755,328 || trainable%: 0.1984
Starting training...


Step,Training Loss
5,2.583301
10,2.115247
15,1.859994
20,1.726392
25,1.589399
30,1.515964
35,1.397714
40,1.257980
45,1.202659
50,1.169831


Saving LoRA adapter to /kaggle/working/adapters/qwen_lor...
✅ Training successfully completed.


In [9]:
import shutil
import os
adapter_dir = "adapters/qwen_lora"
if os.path.exists(adapter_dir):
    shutil.make_archive("qwen_lora_adapter", 'zip', adapter_dir)
    print("✅ Адаптер успешно упакован в qwen_lora_adapter.zip!")
else:
    print("Папка с адаптером не найдена.")

Папка с адаптером не найдена.
